In [3]:
from google.colab import files
import io
import pandas as pd
import numpy as np
import re
import unicodedata
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.linear_model import LogisticRegression
import warnings

warnings.filterwarnings('ignore')

print("Silakan klik 'Choose Files' di bawah ini untuk mengupload file CSV Anda:")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
print(f"\nFile '{file_name}' berhasil diunggah! Memproses data...")

df = pd.read_csv(io.BytesIO(uploaded[file_name]), encoding='utf-8-sig')

def clean_youtube_comment_fixed(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = unicodedata.normalize('NFKC', text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.strip()

df['Clean_Comment'] = df['Comment'].apply(clean_youtube_comment_fixed)

def encode_labels(label):
    if pd.isna(label) or str(label).strip() == '':
        return -1
    val = str(label).strip().upper()
    if val == 'P':
        return 1
    elif val == 'NP':
        return 0
    else:
        return -1

df['Label_Encoded'] = df['Class'].apply(encode_labels)
y = df['Label_Encoded'].values

stopwords_id = ['yang', 'di', 'ke', 'dari', 'pada', 'dalam', 'untuk', 'dengan', 'dan', 'atau', 'ini', 'itu', 'juga', 'sudah', 'saya', 'dia', 'mereka', 'kita', 'kami', 'kamu', 'ada', 'tidak', 'bisa', 'akan', 'aku', 'buat', 'sama', 'yg', 'aja', 'nya']
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.8, stop_words=stopwords_id)
X = vectorizer.fit_transform(df['Clean_Comment'])

base_classifier = LogisticRegression(class_weight='balanced', max_iter=1000)
self_training_model = SelfTrainingClassifier(base_classifier, threshold=0.85, max_iter=10, verbose=True)

print("\nMemulai proses pelatihan AI (Self-Training)...")
self_training_model.fit(X, y)
print("Pelatihan selesai!\n")

unlabeled_mask = df['Label_Encoded'] == -1
X_unlabeled = X[unlabeled_mask.to_numpy()]

predictions = self_training_model.predict(X_unlabeled)
pred_labels_text = ['P' if p == 1 else 'NP' for p in predictions]

df['Final_Class'] = df['Class'].str.strip().str.upper()
df.loc[unlabeled_mask, 'Final_Class'] = pred_labels_text

df_final = df[['No', 'Comment Id', 'Comment', 'Class', 'Final_Class']]

print("--- View Shot: Hasil Pelabelan yang Baru ---")
display(df_final[unlabeled_mask].head(10))

output_filename = 'Dataset_JudiOnline_Tepat.csv'
df_final.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"\nSelesai! Menyiapkan file {output_filename} untuk diunduh...")
files.download(output_filename)

Silakan klik 'Choose Files' di bawah ini untuk mengupload file CSV Anda:


Saving Dataset_Youtube_Online Gambling Promotions.csv to Dataset_Youtube_Online Gambling Promotions (1).csv

File 'Dataset_Youtube_Online Gambling Promotions (1).csv' berhasil diunggah! Memproses data...

Memulai proses pelatihan AI (Self-Training)...
End of iteration 1, added 86 new labels.
End of iteration 2, added 42 new labels.
End of iteration 3, added 24 new labels.
End of iteration 4, added 28 new labels.
End of iteration 5, added 34 new labels.
End of iteration 6, added 43 new labels.
End of iteration 7, added 91 new labels.
End of iteration 8, added 79 new labels.
End of iteration 9, added 67 new labels.
End of iteration 10, added 68 new labels.
Pelatihan selesai!

--- View Shot: Hasil Pelabelan yang Baru ---


,No,Comment Id,Comment,Class,Final_Class
1011,1012.0,Ugy6ZVdS2zeQaPqJMiF4AaABAg,"Asli, Ketemu Alexis17 di game, ternyata orangn...",NaN,P
1012,1013.0,UgypZa41VbQd-cr26up4AaABAg,"Solid banget timnya Alexis17, pantesan sering ...",NaN,P
1013,1014.0,Ugyc2wiIkLSZAuSZLUx4AaABAg,Ada yang kenal Alexis17? Mau ajak download gam...,NaN,P
1014,1015.0,UgwNwtF97VEJ2YqTDrd4AaABAg,"Damage ALEXIS17 gak ada obat, musuh auto rata....",NaN,P
1015,1016.0,Ugx0d6oTdDJ46xMF_Wh4AaABAg,"Info mabar dong, alexis17 lagi nyari temen nih.",NaN,P
1016,1017.0,UgyM1EqgFMCXOhlYkMJ4AaABAg,Hp kentang gini mana kuat kayak 𝐀𝐋𝐄𝐗𝐈𝐒17 pas n...,NaN,P
1017,1018.0,UgzpAB8tozpUJg6efHN4AaABAg,Ada yang kenal ALEXIS17? Mau ajak beli skin.,NaN,P
1018,1019.0,Ugwo7PFrlv4yDjTpi-x4AaABAg,"Pengen punya PC kayak ALEXIS17, biar nge-game ...",NaN,P
1019,1020.0,UgwbmRWZBeb0G_oSZDZ4AaABAg,Ada yang kenal alexis17? Mau ajak top up.,NaN,P
1020,1021.0,Ugzvlkgf-ZX-haf8NXd4AaABAg,"Sumpah, Solid banget timnya Alexis17, pantesan...",NaN,P



Selesai! Menyiapkan file Dataset_JudiOnline_Tepat.csv untuk diunduh...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>